# 00 — Comparación Raw vs Clean
Evalúa el impacto del preprocesado de audio (filtro Wiener + eliminación de clipping) sobre las predicciones del modelo YOLO.

**Hipótesis**: el audio contaminado genera falsos positivos → la limpieza reduce eventos espurios y aumenta la confianza media en eventos reales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats as sp_stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

CLASSES = ['Horn', 'Siren', 'Speech', 'Ringtone', 'Cry', 'Pets', 'Physiological', 'Vibration', 'Alarma']

data_path = Path('../data/processed')
raw_dir   = Path('../data/raw')
Path('../outputs').mkdir(parents=True, exist_ok=True)

# --- Cargar raw (.txt de cada día) ---
COLS_RAW = ['microfono_id', 't_start', 't_end', 'class', 'confidence']
dfs_raw = []
for f in raw_dir.rglob('predicciones*.txt'):
    df = pd.read_csv(f, header=None, names=COLS_RAW, on_bad_lines='skip')
    df['date'] = f.parent.name
    dfs_raw.append(df)
raw = pd.concat(dfs_raw, ignore_index=True)
raw['t_start'] = pd.to_datetime(raw['t_start'], format='mixed', errors='coerce')
raw = raw.dropna(subset=['t_start'])
raw['class'] = pd.to_numeric(raw['class'], errors='coerce').astype('Int64')
raw['class_name'] = raw['class'].apply(lambda x: CLASSES[int(x)] if pd.notna(x) and 0<=int(x)<len(CLASSES) else 'Unknown')

# --- Cargar clean (predicciones_clean.csv) ---
clean = pd.read_csv(data_path / 'predicciones_clean.csv')
clean = clean.rename(columns={'mic_id':'microfono_id','timestamp_onset':'t_start',
                               'timestamp_offset':'t_end','class_id':'class'})
clean['t_start'] = pd.to_datetime(clean['t_start'], format='mixed', errors='coerce')
clean = clean.dropna(subset=['t_start'])
clean['date'] = clean['t_start'].dt.strftime('%d-%m-%Y')
clean['class'] = pd.to_numeric(clean['class'], errors='coerce').astype('Int64')
clean['class_name'] = clean['class'].apply(lambda x: CLASSES[int(x)] if pd.notna(x) and 0<=int(x)<len(CLASSES) else 'Unknown')

shared_dates = sorted(set(raw['date'].unique()) & set(clean['date'].unique()))
print(f'RAW  total: {len(raw):,} eventos en {raw["date"].nunique()} días')
print(f'CLEAN total: {len(clean):,} eventos en {clean["date"].nunique()} días')
print(f'Días compartidos: {len(shared_dates)} → {shared_dates}')
print(f'Solo en RAW:      {sorted(set(raw["date"].unique()) - set(clean["date"].unique()))}')
print(f'Solo en CLEAN:    {sorted(set(clean["date"].unique()) - set(raw["date"].unique()))}')


## 1. Cobertura de Fechas
¿Qué días tienen datos raw, clean o ambos? Los días solo-clean son grabaciones antiguas recién procesadas. Los días solo-raw necesitan aún limpieza de audio.

In [ ]:
all_dates = sorted(set(raw['date'].unique()) | set(clean['date'].unique()))
raw_counts   = raw.groupby('date').size().rename('raw')
clean_counts = clean.groupby('date').size().rename('clean')
df_dates = pd.DataFrame({'raw': raw_counts, 'clean': clean_counts}).reindex(all_dates).fillna(0).astype(int)
df_dates['solo_raw']   = (df_dates['raw'] > 0) & (df_dates['clean'] == 0)
df_dates['solo_clean'] = (df_dates['clean'] > 0) & (df_dates['raw'] == 0)
df_dates['ambos']      = (df_dates['raw'] > 0) & (df_dates['clean'] > 0)
df_dates['delta']      = df_dates['clean'] - df_dates['raw']
df_dates['delta_pct']  = ((df_dates['clean'] - df_dates['raw']) / df_dates['raw'].replace(0, np.nan) * 100).round(1)

print('Cobertura por fecha:')
display(df_dates[['raw','clean','delta','delta_pct']].rename(columns={'delta':'Δ eventos','delta_pct':'Δ %'}))

# Mapa de cobertura visual
fig, ax = plt.subplots(figsize=(13, 3))
x = range(len(all_dates))
ax.bar(x, df_dates['raw'],   label='Raw',   alpha=0.6, color='tomato')
ax.bar(x, df_dates['clean'], label='Clean', alpha=0.7, color='steelblue', width=0.5)
ax.set_xticks(list(x))
ax.set_xticklabels([d[:-5] for d in all_dates], rotation=45, ha='right')
ax.set_title('Eventos por Día: Raw vs Clean')
ax.set_ylabel('Nº eventos')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/raw_vs_clean_by_date.png', bbox_inches='tight')
plt.show()


## 2. Impacto Total — Solo Días Compartidos
Comparamos en los días donde tenemos AMBAS fuentes para una evaluación justa del impacto de la limpieza.

In [ ]:
raw_s   = raw[raw['date'].isin(shared_dates)].copy()
clean_s = clean[clean['date'].isin(shared_dates)].copy()

print(f'=== DÍAS COMPARTIDOS ({len(shared_dates)} días) ===')
print(f'Raw   eventos: {len(raw_s):,}')
print(f'Clean eventos: {len(clean_s):,}')
delta_total = len(clean_s) - len(raw_s)
delta_pct   = delta_total / len(raw_s) * 100
print(f'Δ = {delta_total:+,} ({delta_pct:+.1f}%)')
print()
print(f'Confianza media — Raw: {raw_s["confidence"].mean():.4f} | Clean: {clean_s["confidence"].mean():.4f}')
t_stat, p_val = sp_stats.mannwhitneyu(clean_s['confidence'].dropna(), raw_s['confidence'].dropna(), alternative='two-sided')
print(f'Mann-Whitney U confianza (p={p_val:.2e}): {"diferencia significativa" if p_val < 0.05 else "sin diferencia significativa"}')


## 3. Cambio por Clase
¿Qué clases perdieron más eventos tras la limpieza? Un descenso grande sugiere falsos positivos por ruido en esa clase.

In [ ]:
raw_cls   = raw_s.groupby('class_name').size().rename('raw')
clean_cls = clean_s.groupby('class_name').size().rename('clean')
df_cls = pd.DataFrame({'raw': raw_cls, 'clean': clean_cls}).reindex(CLASSES, fill_value=0)
df_cls['delta']     = df_cls['clean'] - df_cls['raw']
df_cls['delta_pct'] = (df_cls['delta'] / df_cls['raw'].replace(0, np.nan) * 100).round(1)
df_cls['reduccion'] = df_cls['delta_pct'] < -10

display(df_cls[['raw','clean','delta','delta_pct']].rename(columns={'delta':'Δ','delta_pct':'Δ %'}))

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(CLASSES))
w = 0.38
axes[0].bar(x - w/2, df_cls['raw'],   width=w, label='Raw',   color='tomato',    alpha=0.8)
axes[0].bar(x + w/2, df_cls['clean'], width=w, label='Clean', color='steelblue', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[0].set_title('Eventos por Clase: Raw vs Clean')
axes[0].set_ylabel('Nº eventos')
axes[0].legend()

colors = ['green' if d >= 0 else 'red' for d in df_cls['delta_pct'].fillna(0)]
axes[1].barh(CLASSES, df_cls['delta_pct'].fillna(0), color=colors)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Variación % por Clase tras Limpieza')
axes[1].set_xlabel('Δ %')
for i, v in enumerate(df_cls['delta_pct'].fillna(0)):
    axes[1].text(v + (1 if v >= 0 else -1), i, f'{v:+.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/raw_vs_clean_by_class.png', bbox_inches='tight')
plt.show()


## 4. Distribución de Confianza
Si la limpieza elimina ruido, esperamos: ↑ confianza media en eventos conservados, ↓ cola de baja confianza.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma global
axes[0].hist(raw_s['confidence'].dropna(),   bins=40, alpha=0.6, color='tomato',    label=f'Raw (n={len(raw_s):,})',   density=True)
axes[0].hist(clean_s['confidence'].dropna(), bins=40, alpha=0.6, color='steelblue', label=f'Clean (n={len(clean_s):,})', density=True)
axes[0].axvline(raw_s['confidence'].mean(),   color='red',  linestyle='--', lw=1.5, label=f'Media raw={raw_s["confidence"].mean():.3f}')
axes[0].axvline(clean_s['confidence'].mean(), color='blue', linestyle='--', lw=1.5, label=f'Media clean={clean_s["confidence"].mean():.3f}')
axes[0].set_title('Distribución de Confianza Global')
axes[0].set_xlabel('Confianza')
axes[0].set_ylabel('Densidad')
axes[0].legend(fontsize=9)

# Boxplot de confianza por clase
raw_s_plot   = raw_s.copy();   raw_s_plot['fuente'] = 'Raw'
clean_s_plot = clean_s.copy(); clean_s_plot['fuente'] = 'Clean'
combined = pd.concat([raw_s_plot, clean_s_plot])
sns.boxplot(data=combined, x='class_name', y='confidence', hue='fuente',
            palette={'Raw':'tomato','Clean':'steelblue'}, ax=axes[1],
            order=CLASSES, fliersize=2)
axes[1].set_title('Confianza por Clase: Raw vs Clean')
axes[1].set_xlabel('')
axes[1].set_ylabel('Confianza')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Fuente')

plt.tight_layout()
plt.savefig('../outputs/raw_vs_clean_confidence.png', bbox_inches='tight')
plt.show()


## 5. Análisis por Día — Impacto de la Limpieza
¿Qué días se beneficiaron más de la limpieza? Un Δ negativo grande indica que ese día tenía mucho ruido de grabación.

In [ ]:
# Confianza media por día y fuente
raw_daily   = raw_s.groupby('date')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_raw','count':'n_raw'})
clean_daily = clean_s.groupby('date')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_clean','count':'n_clean'})
daily = pd.concat([raw_daily, clean_daily], axis=1).reindex(sorted(shared_dates))
daily['conf_delta'] = (daily['conf_clean'] - daily['conf_raw']).round(4)
daily['n_delta_pct'] = ((daily['n_clean'] - daily['n_raw']) / daily['n_raw'] * 100).round(1)
display(daily.rename(columns={'conf_raw':'Conf Raw','conf_clean':'Conf Clean','conf_delta':'ΔConf',
                               'n_raw':'N Raw','n_clean':'N Clean','n_delta_pct':'ΔN %'}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dates_short = [d[:-5] for d in sorted(shared_dates)]

axes[0].bar(dates_short, daily['n_delta_pct'],
            color=['green' if x >= 0 else 'red' for x in daily['n_delta_pct']])
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Variación % en Nº Eventos por Día')
axes[0].set_ylabel('Δ %')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(dates_short, daily['conf_delta'],
            color=['green' if x >= 0 else 'red' for x in daily['conf_delta']])
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Variación en Confianza Media por Día')
axes[1].set_ylabel('Δ Confianza')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/raw_vs_clean_daily.png', bbox_inches='tight')
plt.show()


## 6. Zoom: Días con Mayor Impacto
Para los días con mayor reducción de eventos, ¿qué clases desaparecieron? Identifica si el ruido afecta más a ciertas frecuencias acústicas.

In [ ]:
# Día con mayor reducción porcentual
biggest_drop_date = daily['n_delta_pct'].idxmin()
print(f'Día con mayor reducción: {biggest_drop_date} ({daily.loc[biggest_drop_date, "n_delta_pct"]:.1f}%)')

for focus_date in [biggest_drop_date]:
    r_day = raw_s[raw_s['date'] == focus_date]
    c_day = clean_s[clean_s['date'] == focus_date]
    r_cls = r_day.groupby('class_name').size().rename('raw')
    c_cls = c_day.groupby('class_name').size().rename('clean')
    df_day = pd.DataFrame({'raw': r_cls, 'clean': c_cls}).reindex(CLASSES, fill_value=0)
    df_day['delta_pct'] = ((df_day['clean'] - df_day['raw']) / df_day['raw'].replace(0, np.nan) * 100).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(CLASSES))
    w = 0.38
    axes[0].bar(x - w/2, df_day['raw'],   width=w, color='tomato',    alpha=0.8, label='Raw')
    axes[0].bar(x + w/2, df_day['clean'], width=w, color='steelblue', alpha=0.8, label='Clean')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
    axes[0].set_title(f'{focus_date} — Eventos por Clase')
    axes[0].legend()

    colors = ['green' if v >= 0 else 'red' for v in df_day['delta_pct'].fillna(0)]
    axes[1].barh(CLASSES, df_day['delta_pct'].fillna(0), color=colors)
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].set_title(f'{focus_date} — Variación % por Clase')
    axes[1].set_xlabel('Δ %')
    plt.tight_layout()
    plt.savefig(f'../outputs/raw_vs_clean_{focus_date}.png', bbox_inches='tight')
    plt.show()

    print(f'\n{focus_date}:')
    print(f'  Raw: {len(r_day)} | Clean: {len(c_day)} | Conf media raw={r_day["confidence"].mean():.3f} | clean={c_day["confidence"].mean():.3f}')
    display(df_day[['raw','clean','delta_pct']].rename(columns={'delta_pct':'Δ %'}))


## 7. Días Solo en RAW — Pendientes de Limpiar
Estos días tienen predicciones raw pero NO tienen versión limpia. Se recomienda procesar el audio con `scripts/clean_audio_prueba.py` y re-ejecutar `scripts/infer_clean.py` para incluirlos en el análisis.

In [ ]:
only_raw = sorted(set(raw['date'].unique()) - set(clean['date'].unique()))
only_clean = sorted(set(clean['date'].unique()) - set(raw['date'].unique()))

print('=== Días SOLO en RAW (necesitan audio cleaning) ===')
if only_raw:
    for d in only_raw:
        n = len(raw[raw['date'] == d])
        print(f'  {d}: {n} eventos raw sin limpiar')
    print(f'  Total eventos no procesados: {len(raw[raw["date"].isin(only_raw)])}')
    print()
    print('Para procesarlos:')
    print('  1. Localizar WAV de esos días en data/clean/ o data/raw/')
    print('  2. python scripts/clean_audio_prueba.py  (sobre esos WAVs)')
    print('  3. python scripts/infer_clean.py         (re-genera predicciones_clean.csv)')
    print('  4. Re-ejecutar 01_etl.ipynb')
else:
    print('  Ninguno — todos los días raw tienen versión clean.')

print()
print('=== Días SOLO en CLEAN (sin GPS disponible) ===')
if only_clean:
    for d in only_clean:
        n = len(clean[clean['date'] == d])
        print(f'  {d}: {n} eventos clean (sin track GPS → no geolocalizables)')
else:
    print('  Ninguno.')


## 8. Resumen Ejecutivo
Tabla resumen para incluir en la memoria del TFG.

In [ ]:
summary_data = {
    'Métrica': [
        'Días con datos raw',
        'Días con datos clean',
        'Días compartidos (comparables)',
        'Eventos raw (días compartidos)',
        'Eventos clean (días compartidos)',
        'Variación total (Δ eventos)',
        'Variación total (%)',
        'Confianza media raw',
        'Confianza media clean',
        'Variación confianza (Δ)',
        'Test Mann-Whitney (p-value)',
        'Diferencia significativa',
    ],
    'Valor': [
        raw['date'].nunique(),
        clean['date'].nunique(),
        len(shared_dates),
        len(raw_s),
        len(clean_s),
        f'{len(clean_s) - len(raw_s):+,}',
        f'{(len(clean_s) - len(raw_s)) / len(raw_s) * 100:+.1f}%',
        f'{raw_s["confidence"].mean():.4f}',
        f'{clean_s["confidence"].mean():.4f}',
        f'{clean_s["confidence"].mean() - raw_s["confidence"].mean():+.4f}',
        f'{sp_stats.mannwhitneyu(clean_s["confidence"].dropna(), raw_s["confidence"].dropna(), alternative="two-sided")[1]:.2e}',
        'Sí (p<0.05)' if sp_stats.mannwhitneyu(clean_s['confidence'].dropna(), raw_s['confidence'].dropna(), alternative='two-sided')[1] < 0.05 else 'No',
    ]
}
df_summary = pd.DataFrame(summary_data)
display(df_summary.style.set_properties(**{'text-align': 'left'}))
